# 06 — Performance Tracking & Dashboard

This notebook tracks Claude's recommendation accuracy over time:
- Loads all historical analysis reports from `reports/`
- Compares recommendations against actual subsequent price movement
- Computes hit rate, average return by action type, and confidence calibration
- Produces a visual dashboard of portfolio value and recommendation quality

**Prerequisites:**
- At least one analysis report in `reports/` (from notebook 04)
- Best results with multiple reports over days/weeks

## Setup

In [1]:
import json
import os
import glob
from datetime import datetime, timedelta

import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

REPORTS_DIR = os.path.join("..", "reports")

# Load all analysis reports (excluding latest symlink)
report_files = sorted(glob.glob(os.path.join(REPORTS_DIR, "analysis_*.json")))

reports = []
for f in report_files:
    with open(f, "r") as fh:
        data = json.load(fh)
        data["_filename"] = os.path.basename(f)
        reports.append(data)

print(f"✅ Loaded {len(reports)} analysis report(s)")
for r in reports:
    recs = r.get("analysis", {}).get("recommendations", [])
    print(f"   {r['_filename']}: {len(recs)} recommendations ({r.get('model', '?')})")

✅ Loaded 17 analysis report(s)
   analysis_2026-02-16_2040.json: 9 recommendations (claude-sonnet-4-20250514)
   analysis_2026-02-18_1615.json: 19 recommendations (claude-sonnet-4-20250514)
   analysis_2026-02-18_1657.json: 63 recommendations (claude-opus-4-6)
   analysis_2026-02-18_2118.json: 66 recommendations (claude-sonnet-4-6)
   analysis_2026-02-19_1850.json: 61 recommendations (claude-sonnet-4-6)
   analysis_2026-02-24_0806.json: 52 recommendations (claude-sonnet-4-6)
   analysis_2026-02-25_2314.json: 53 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_0805.json: 51 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_0932.json: 52 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_1022.json: 55 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_1038.json: 51 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_2140.json: 56 recommendations (claude-sonnet-4-6)
   analysis_2026-02-26_2153.json: 55 recommendations (claude-sonnet-4-6)
   analys

## Extract All Recommendations Into a DataFrame

In [2]:
all_recs = []

for report in reports:
    report_date = report.get("generated_at", "")[:10]
    model = report.get("model", "unknown")
    analysis = report.get("analysis", {})
    
    for rec in analysis.get("recommendations", []):
        all_recs.append({
            "report_date": report_date,
            "model": model,
            "ticker": rec.get("ticker"),
            "name": rec.get("name", "")[:30],
            "action": rec.get("action"),
            "confidence": rec.get("confidence"),
            "urgency": rec.get("urgency"),
            "thesis": rec.get("thesis", "")[:100],
        })

recs_df = pd.DataFrame(all_recs)
print(f"📊 Total recommendations across all reports: {len(recs_df)}")

if not recs_df.empty:
    print(f"\nBreakdown by action:")
    print(recs_df["action"].value_counts().to_string())
    print(f"\nBreakdown by confidence:")
    print(recs_df["confidence"].value_counts().to_string())

📊 Total recommendations across all reports: 851

Breakdown by action:
action
HOLD    682
SELL     89
BUY      80

Breakdown by confidence:
confidence
medium    487
high      203
low       161


## Measure Recommendation Accuracy

For each recommendation, we check the actual price change over the following:
- **1 day** — immediate reaction
- **5 days** — one week
- **21 days** — one month

A recommendation is "correct" if:
- BUY → price went up
- SELL → price went down
- HOLD → price stayed within ±5%

In [3]:
def get_forward_returns(ticker: str, from_date: str, periods: list[int]) -> dict:
    """Get actual forward returns for a ticker from a given date."""
    try:
        start = datetime.strptime(from_date, "%Y-%m-%d")
        end = start + timedelta(days=max(periods) + 10)  # buffer for weekends
        
        df = yf.download(
            ticker,
            start=start.strftime("%Y-%m-%d"),
            end=min(end, datetime.now()).strftime("%Y-%m-%d"),
            progress=False,
            auto_adjust=True,
        )
        
        if df.empty or len(df) < 2:
            return {}
        
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        base_price = df["Close"].iloc[0]
        returns = {}
        
        for p in periods:
            if len(df) > p:
                future_price = df["Close"].iloc[p]
                returns[f"return_{p}d"] = round(float((future_price / base_price - 1) * 100), 2)
        
        return returns
    except Exception:
        return {}


# Only measure if we have recommendations with valid dates
if not recs_df.empty:
    print("Fetching forward returns for all recommendations...\n")
    
    periods = [1, 5, 21]
    forward_data = []
    
    unique_combos = recs_df[["ticker", "report_date"]].drop_duplicates()
    
    for _, row in unique_combos.iterrows():
        ticker = row["ticker"]
        date = row["report_date"]
        
        if not ticker or not date:
            continue
        
        returns = get_forward_returns(ticker, date, periods)
        if returns:
            returns["ticker"] = ticker
            returns["report_date"] = date
            forward_data.append(returns)
            status = ", ".join(f"{k}: {v:+.1f}%" for k, v in returns.items() if k.startswith("return"))
            print(f"  {ticker} ({date}): {status}")
    
    if forward_data:
        fwd_df = pd.DataFrame(forward_data)
        recs_df = recs_df.merge(fwd_df, on=["ticker", "report_date"], how="left")
        print(f"\n✅ Forward returns fetched for {len(forward_data)} ticker-date combos")
    else:
        print("\n⚠️  No forward return data available yet (reports may be too recent).")
else:
    print("No recommendations to evaluate.")

Fetching forward returns for all recommendations...

  SBSI (2026-02-16): return_1d: -0.3%, return_5d: -3.5%
  CAMYX (2026-02-16): return_1d: +0.4%, return_5d: +0.0%


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NHX105509"}}}
$NHX105509: possibly delisted; no timezone found

1 Failed download:
['NHX105509']: possibly delisted; no timezone found


  MIPTX (2026-02-16): return_1d: +0.5%, return_5d: +2.6%
  EWZ (2026-02-16): return_1d: +0.8%, return_5d: +4.6%
  BTC (2026-02-16): return_1d: -2.2%, return_5d: -4.8%


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NFLX180201C00355000"}}}
$NFLX180201C00355000: possibly delisted; no timezone found

1 Failed download:
['NFLX180201C00355000']: possibly delisted; no timezone found


  DBLTX (2026-02-16): return_1d: -0.2%, return_5d: +0.1%


...

  VOO (2026-02-25): return_1d: -0.6%
  AAPL (2026-02-25): return_1d: -0.5%
  NVDA (2026-02-25): return_1d: -5.5%
  IXUS (2026-02-25): return_1d: -0.2%
  SPY (2026-02-25): return_1d: -0.6%
  MSFT (2026-02-25): return_1d: +0.3%
  GLTR (2026-02-25): return_1d: +0.7%
  SFY (2026-02-25): return_1d: -1.0%
  QQQ (2026-02-25): return_1d: -1.2%
  ICLN (2026-02-25): return_1d: -2.2%
  AMZN (2026-02-25): return_1d: -1.3%
  V (2026-02-25): return_1d: +1.2%
  VOX (2026-02-25): return_1d: +0.1%
  MA (2026-02-25): return_1d: +1.1%
  GAMR (2026-02-25): return_1d: +0.2%
  IEFA (2026-02-25): return_1d: -0.1%
  NFLX (2026-02-25): return_1d: +2.3%
  VWO (2026-02-25): return_1d: -1.2%
  VPL (2026-02-25): return_1d: +0.2%
  IJH (2026-02-25): return_1d: +0.4%
  MILN (2026-02-25): return_1d: +1.6%
  INDA (2026-02-25): return_1d: -0.3%
  VEA (2026-02-25): return_1d: +0.1%
  DVN (2026-02-25): return_1d: +0.1%
  R (2026-02-25): return_1d: +1.0%
  DYNF (2026-02-25): return_1d: -0.8%
  IVW (2026-02-25): return_1d:

$BRK.B: possibly delisted; no timezone found

1 Failed download:
['BRK.B']: possibly delisted; no timezone found


  VOX (2026-02-26): return_1d: +0.8%
  VTI (2026-02-26): return_1d: -0.5%
  LQD (2026-02-26): return_1d: -0.0%
  VWOB (2026-02-26): return_1d: +0.0%
  VNQ (2026-02-26): return_1d: +0.2%


$BRK.B: possibly delisted; no timezone found

1 Failed download:
['BRK.B']: possibly delisted; no timezone found
$SOFI: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)

1 Failed download:
['SOFI']: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)
$MSFT: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)

1 Failed download:
['MSFT']: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)
$NVDA: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)

1 Failed download:
['NVDA']: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)
$AAPL: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)

1 Failed download:
['AAPL']: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)
$AMZN: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)

1 Failed download:
['AMZN']: possibly delisted; no price data found  (1d 2026-03-01 -> 2026-03-02)
$V


✅ Forward returns fetched for 285 ticker-date combos


## Accuracy Scorecard

In [4]:
def score_recommendation(row, return_col="return_5d"):
    """Score whether a recommendation was correct."""
    ret = row.get(return_col)
    action = row.get("action")
    
    if ret is None or pd.isna(ret):
        return None
    
    if action == "BUY":
        return "correct" if ret > 0 else "incorrect"
    elif action == "SELL":
        return "correct" if ret < 0 else "incorrect"
    elif action == "HOLD":
        return "correct" if abs(ret) <= 5 else "incorrect"
    return None


if "return_5d" in recs_df.columns:
    for period in ["return_1d", "return_5d", "return_21d"]:
        if period in recs_df.columns:
            recs_df[f"score_{period}"] = recs_df.apply(
                lambda r: score_recommendation(r, period), axis=1
            )
    
    print("═" * 60)
    print("         RECOMMENDATION ACCURACY SCORECARD")
    print("═" * 60)
    
    for period_label, col in [("1-Day", "score_return_1d"), ("5-Day", "score_return_5d"), ("21-Day", "score_return_21d")]:
        if col in recs_df.columns:
            scored = recs_df[col].dropna()
            if len(scored) > 0:
                correct = (scored == "correct").sum()
                total = len(scored)
                pct = correct / total * 100
                print(f"  {period_label:8s}: {correct}/{total} correct ({pct:.0f}%)")
            else:
                print(f"  {period_label:8s}: No data yet")
    
    print("═" * 60)
    
    # Accuracy by action type
    if "score_return_5d" in recs_df.columns:
        print("\n📊 5-Day Accuracy by Action:")
        for action in ["BUY", "SELL", "HOLD"]:
            subset = recs_df[recs_df["action"] == action]["score_return_5d"].dropna()
            if len(subset) > 0:
                correct = (subset == "correct").sum()
                emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡"}[action]
                print(f"  {emoji} {action:4s}: {correct}/{len(subset)} ({correct/len(subset)*100:.0f}%)")
    
    # Accuracy by confidence level
    if "score_return_5d" in recs_df.columns:
        print("\n📊 5-Day Accuracy by Confidence:")
        for conf in ["high", "medium", "low"]:
            subset = recs_df[recs_df["confidence"] == conf]["score_return_5d"].dropna()
            if len(subset) > 0:
                correct = (subset == "correct").sum()
                print(f"  {conf:6s}: {correct}/{len(subset)} ({correct/len(subset)*100:.0f}%)")
else:
    print("⚠️  No forward return data yet. Run again after market data is available for your report dates.")
    print("   (Reports from today won't have forward returns until tomorrow.)")

════════════════════════════════════════════════════════════
         RECOMMENDATION ACCURACY SCORECARD
════════════════════════════════════════════════════════════
  1-Day   : 616/685 correct (90%)
  5-Day   : 174/212 correct (82%)
════════════════════════════════════════════════════════════

📊 5-Day Accuracy by Action:
  🟢 BUY : 12/21 (57%)
  🔴 SELL: 23/47 (49%)
  🟡 HOLD: 139/144 (97%)

📊 5-Day Accuracy by Confidence:
  high  : 37/62 (60%)
  medium: 103/114 (90%)
  low   : 34/36 (94%)


## Detailed Results Table

In [5]:
display_cols = ["report_date", "ticker", "action", "confidence"]

for col in ["return_1d", "return_5d", "return_21d", "score_return_5d"]:
    if col in recs_df.columns:
        display_cols.append(col)

if not recs_df.empty:
    recs_df[display_cols].sort_values(["report_date", "action"], ascending=[False, True])
else:
    print("No recommendations to display.")

## Average Return by Action Type

If Claude's recommendations are good, BUY picks should have positive returns and SELL picks negative.

In [6]:
return_cols = [c for c in ["return_1d", "return_5d", "return_21d"] if c in recs_df.columns]

if return_cols and not recs_df.empty:
    print("📈 Average Forward Return by Action Type:\n")
    
    avg_returns = recs_df.groupby("action")[return_cols].mean().round(2)
    
    for action in ["BUY", "HOLD", "SELL"]:
        if action in avg_returns.index:
            emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡"}[action]
            returns_str = "  |  ".join(
                f"{col.replace('return_', '')}: {avg_returns.loc[action, col]:+.2f}%"
                for col in return_cols
            )
            print(f"  {emoji} {action:4s}:  {returns_str}")
    
    print("\n  (Positive returns on BUYs and negative on SELLs = good signals)")
else:
    print("Not enough data yet for return analysis.")

📈 Average Forward Return by Action Type:

  🟢 BUY :  1d: +0.33%  |  5d: +1.16%
  🟡 HOLD:  1d: +0.01%  |  5d: +0.80%
  🔴 SELL:  1d: -0.83%  |  5d: -0.90%

  (Positive returns on BUYs and negative on SELLs = good signals)


## Portfolio Value Over Time

Track total portfolio value across analysis runs.

In [7]:
portfolio_history = []

for report in reports:
    generated = report.get("generated_at", "")[:16]
    summary = report.get("analysis", {}).get("portfolio_assessment", {})
    
    # Try to get portfolio value from the enriched data if we saved it
    # For now, extract what's in the report
    recs = report.get("analysis", {}).get("recommendations", [])
    
    portfolio_history.append({
        "date": generated,
        "health": summary.get("overall_health", "N/A"),
        "risk": summary.get("risk_level", "N/A"),
        "num_buys": sum(1 for r in recs if r.get("action") == "BUY"),
        "num_sells": sum(1 for r in recs if r.get("action") == "SELL"),
        "num_holds": sum(1 for r in recs if r.get("action") == "HOLD"),
        "model": report.get("model", "?"),
        "input_tokens": report.get("usage", {}).get("input_tokens", 0),
        "output_tokens": report.get("usage", {}).get("output_tokens", 0),
    })

history_df = pd.DataFrame(portfolio_history)

if not history_df.empty:
    print("📅 Analysis History:\n")
    print(history_df.to_string(index=False))
else:
    print("No history yet.")

📅 Analysis History:

            date   health     risk  num_buys  num_sells  num_holds                    model  input_tokens  output_tokens
2026-02-16T20:40     weak     high         0          5          4 claude-sonnet-4-20250514         11062           2448
2026-02-18T16:15 moderate moderate         4          6          9 claude-sonnet-4-20250514         53311           4529
2026-02-18T16:57 moderate moderate         6         11         46          claude-opus-4-6         53458          13802
2026-02-18T21:18 moderate moderate         6         15         45        claude-sonnet-4-6         53458          16924
2026-02-19T18:50 moderate moderate         5         12         44        claude-sonnet-4-6         51887          15220
2026-02-24T08:06 moderate moderate         4          2         46        claude-sonnet-4-6         43839          16215
2026-02-25T23:14 moderate moderate         5          2         46        claude-sonnet-4-6         43822          13837
2026-02-26T

## API Usage & Cost Tracking

In [8]:
# Approximate pricing (check https://docs.anthropic.com for current rates)
PRICING = {
    "claude-sonnet-4-20250514": {"input": 3.0, "output": 15.0},  # per 1M tokens
    "claude-opus-4-5-20250514": {"input": 15.0, "output": 75.0},
}

total_input_tokens = 0
total_output_tokens = 0
total_cost = 0

print("═" * 60)
print("         API USAGE & COST SUMMARY")
print("═" * 60)

for report in reports:
    usage = report.get("usage", {})
    model = report.get("model", "")
    inp = usage.get("input_tokens", 0)
    out = usage.get("output_tokens", 0)
    
    # Find matching pricing (partial match)
    rates = {"input": 3.0, "output": 15.0}  # default to sonnet
    for model_key, model_rates in PRICING.items():
        if model_key in model:
            rates = model_rates
            break
    
    cost = (inp / 1_000_000 * rates["input"]) + (out / 1_000_000 * rates["output"])
    
    total_input_tokens += inp
    total_output_tokens += out
    total_cost += cost
    
    print(f"  {report.get('generated_at', '')[:10]}  {model[:30]:30s}  {inp:>7,} in  {out:>6,} out  ${cost:.4f}")

print(f"{'─' * 60}")
print(f"  {'TOTAL':42s}  {total_input_tokens:>7,} in  {total_output_tokens:>6,} out  ${total_cost:.4f}")
print(f"\n  📊 Avg cost per analysis: ${total_cost / max(len(reports), 1):.4f}")
print(f"  📅 Projected monthly (daily weekday runs): ${total_cost / max(len(reports), 1) * 22:.2f}")
print("═" * 60)

════════════════════════════════════════════════════════════
         API USAGE & COST SUMMARY
════════════════════════════════════════════════════════════
  2026-02-16  claude-sonnet-4-20250514         11,062 in   2,448 out  $0.0699
  2026-02-18  claude-sonnet-4-20250514         53,311 in   4,529 out  $0.2279
  2026-02-18  claude-opus-4-6                  53,458 in  13,802 out  $0.3674
  2026-02-18  claude-sonnet-4-6                53,458 in  16,924 out  $0.4142
  2026-02-19  claude-sonnet-4-6                51,887 in  15,220 out  $0.3840
  2026-02-24  claude-sonnet-4-6                43,839 in  16,215 out  $0.3747
  2026-02-25  claude-sonnet-4-6                43,822 in  13,837 out  $0.3390
  2026-02-26  claude-sonnet-4-6                43,817 in  13,579 out  $0.3351
  2026-02-26  claude-sonnet-4-6                43,815 in  13,663 out  $0.3364
  2026-02-26  claude-sonnet-4-6                46,765 in  14,283 out  $0.3545
  2026-02-26  claude-sonnet-4-6                46,773 in  14,236

## Recommendation Consistency

Track how Claude's view on each ticker changes across reports.

In [9]:
if len(reports) > 1 and not recs_df.empty:
    print("🔄 Recommendation Changes Over Time:\n")
    
    pivot = recs_df.pivot_table(
        index="ticker",
        columns="report_date",
        values="action",
        aggfunc="first"
    )
    
    # Flag tickers where the recommendation changed
    for ticker in pivot.index:
        actions = pivot.loc[ticker].dropna().tolist()
        if len(set(actions)) > 1:
            change_str = " → ".join(actions)
            print(f"  ⚡ {ticker}: {change_str}")
    
    print("\n  (Changes may indicate evolving market conditions or model inconsistency)")
    
    pivot
else:
    print("Need 2+ reports to track recommendation changes over time.")
    print("Run the pipeline again tomorrow to start building history!")

🔄 Recommendation Changes Over Time:

  ⚡ ADBE: BUY → HOLD → BUY → SELL → BUY
  ⚡ AGG: HOLD → HOLD → SELL → HOLD → HOLD → HOLD → HOLD → HOLD
  ⚡ AMZN: BUY → BUY → HOLD → BUY → BUY → BUY → BUY → BUY
  ⚡ BITO: HOLD → HOLD → HOLD → HOLD → HOLD → SELL → SELL → HOLD
  ⚡ GAMR: HOLD → HOLD → HOLD → HOLD → SELL → SELL → HOLD → SELL
  ⚡ GSK: SELL → SELL → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD
  ⚡ HPK: HOLD → SELL
  ⚡ ICLN: HOLD → HOLD → HOLD → HOLD → HOLD → SELL → SELL
  ⚡ IEFA: HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → BUY
  ⚡ MA: HOLD → HOLD → BUY → BUY → BUY → HOLD → BUY → BUY
  ⚡ MILN: HOLD → SELL → SELL → SELL → SELL → SELL → HOLD → SELL
  ⚡ NFLX: BUY → BUY → HOLD → HOLD → HOLD → SELL → SELL → SELL
  ⚡ NVDA: HOLD → BUY → HOLD → BUY → HOLD → BUY → BUY → HOLD
  ⚡ R: SELL → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD
  ⚡ SHLD: HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → BUY
  ⚡ SOFI: BUY → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD → HOLD
  ⚡ USO: HOLD → HOLD → HOLD → HOLD → HOLD

## Save Performance Log

In [10]:
perf_log = {
    "generated_at": datetime.now().isoformat(),
    "total_reports": len(reports),
    "total_recommendations": len(recs_df),
    "api_cost_total": round(total_cost, 4),
    "recommendations": recs_df.to_dict(orient="records") if not recs_df.empty else [],
    "history": portfolio_history,
}

perf_path = os.path.join(REPORTS_DIR, "performance_log.json")
with open(perf_path, "w") as f:
    json.dump(perf_log, f, indent=2, default=str)

print(f"💾 Performance log saved to {perf_path}")
print(f"\n🎉 Phase 5 complete! Your financial analyst agent is fully operational.")
print(f"")
print(f"   To run daily:")
print(f"   uv run python run_pipeline.py")
print(f"")
print(f"   To schedule via cron (weekdays 7 AM):")
print(f"   0 7 * * 1-5 cd /path/to/financial-agent && uv run python run_pipeline.py >> pipeline.log 2>&1")

💾 Performance log saved to ../reports/performance_log.json

🎉 Phase 5 complete! Your financial analyst agent is fully operational.

   To run daily:
   uv run python run_pipeline.py

   To schedule via cron (weekdays 7 AM):
   0 7 * * 1-5 cd /path/to/financial-agent && uv run python run_pipeline.py >> pipeline.log 2>&1


---

### What's Next?

Your agent is now fully built across all 5 phases:

1. ✅ **Plaid integration** — connect any supported brokerage
2. ✅ **Data enrichment** — prices, technicals, fundamentals, news
3. ✅ **Claude analysis** — structured BUY/SELL/HOLD recommendations
4. ✅ **Notifications** — email, Slack, and/or push delivery
5. ✅ **Performance tracking** — accuracy scoring, cost tracking, consistency monitoring

**To go live with real accounts:** follow the Plaid Production Guide (see README).